## Preprocessing Small Tabular Datasets

In this section, we begin by loading a CSV dataset that contains detailed information from mobile network drive tests. This dataset includes measurements and statistics of thousands of variables related to mobile network performance while a testing device downloads a file of a specific size. Furthermore, we also apply a sequence of preprocessing steps to the loaded dataset to prepare it for the subsequent machine-learning stages. To achieve this, we utilize the pandas library, a well-known tool for conducting mathematical and data management operations on dataframes. It's important to note that dataframes consist of a data type specifically designed for carrying out exploratory data analyses and data preprocessing. In this work, we assume that the virtual environment has been activated and the required libraries have been installed.

## Import libraries

In this context, the first lines of code import **Pandas** and **NumPy** libraries. **Pandas** is a Python package that provides efficient and flexible tools for working with tabular data, such as CSV files or database tables. It allows users to manipulate and analyze data in structures called DataFrames, which are similar to spreadsheets.

**NumPy**, on the other hand, is a library that supports numerical operations and offers functions to work with arrays and matrices. NumPy is essential for performing mathematical calculations on large datasets, which can be particularly useful for data analysis and machine learning tasks.

In [1]:
import pandas as pd
import numpy as np

from statsmodels.stats.outliers_influence import variance_inflation_factor 

import warnings
warnings.filterwarnings('ignore')

<div class="alert alert-block alert-warning">Including a code cell at the beginning of the notebook to import all requisite libraries is advisable, but it is not mandatory.</div>

## Load Dataset

We use the ``pandas.read_csv`` function, which loads the dataset from a CSV file into a table-like structure called a DataFrame. In our case, ``DatasetA.csv`` contains hundreds of measurements and statistics captured during drive tests. Nokia collected this dataset across multiple European countries in 2023. This dataset does not include sensitive information about individuals or operators. 

From this dataset, we filter observations (i.e., data table rows) where the target operator is ``Operator1``, the test type is ``Capacity DL``, and the test qualifier is marked as ``SUCCESS``. In the context of mobile networks, the ``Capacity DL`` test assesses how much data the network can deliver to mobile users during the downlink (data transmitted from the network to the mobile device under test). This test is crucial for evaluating network throughput. 

Our analysis focuses on the throughput variable (labeled ``transfer_datarate``), the target key performance indicator (KPI). We then remove any rows where the throughput value is either missing (NaN) or negative, as these do not provide valid data for analysis. Finally, we select only the columns with numeric data types needed for further statistical analysis or modeling.

In [2]:
print('-----------------------------------------------------------------')
print('LOADING DATASET A FILE DOWNLOAD DATASET')
data_file = "data/DatasetA.csv"
data = pd.read_csv(data_file, encoding = "ISO-8859-1", sep=",", decimal='.', low_memory = False)
print('-----------------------------------------------------------------')

operator  = 'Operator1'
test      = 'Capacity DL'
qualifier = 'SUCCESS'

data = data.loc[ data['home_operator'] == operator]
data = data.loc[ data['data_test_type'] == test]
data = data.loc[ data['qualifier'] == qualifier ]
data = data.loc[~data['transfer_datarate'].isna()]
data = data.loc[data['transfer_datarate'] > 0]
data = data.select_dtypes(['number'])
data.shape

-----------------------------------------------------------------
LOADING DATASET A FILE DOWNLOAD DATASET
-----------------------------------------------------------------


(1001, 659)

## Variable Selection

This stage requires an understanding of the dataset’s structure and variable names. Our study focuses on extracting specific data columns (variables or features) that provide the target KPI and relevant statistics. Specifically, we collect data related to the information transferred from the testing mobile device to the server (``a2b``) and from the server to the testing device (``b2a``). In addition, our code gathers indicator statistics, such as the minimum (``_min``), the 25th percentile 25 (``_25.``), the median (``_50.``), the 75th percentile (``_75.``), the maximum (``_max``), the average (``_avg``), and the sum (``_sum``). After gathering the relevant variables and statistics, we remove certain columns that contain measurements at specific test stages, such as ``VolStep``, ``TimeStep``, ``FirstSec``, and ``FirstMB``. These columns are not necessary for our analysis at this point. We use the ``DataFrame.columns.str.contains()`` function to identify columns. This function allows us to find column names that contain specific strings. Afterward, the code eliminates duplicated columns.

In [3]:
data = data[data.columns[data.columns.str.contains('transfer_datarate')|
                         data.columns.str.contains('.a2b')|
                         data.columns.str.contains('.b2a')|
                         data.columns.str.contains('_min')|
                         data.columns.str.contains('_25.')|
                         data.columns.str.contains('_50.')|
                         data.columns.str.contains('_75.')|
                         data.columns.str.contains('_max')|
                         data.columns.str.contains('_avg')|
                         data.columns.str.contains('_sum')]]

# Remove columns
data = data[data.columns[~data.columns.str.contains('VolStep')]]
data = data[data.columns[~data.columns.str.contains('TimeStep')]]
data = data[data.columns[~data.columns.str.contains('FirstSec')]]
data = data[data.columns[~data.columns.str.contains('FirstMB')]]

print('-----------------------------------------------------------------')
print('Before removing duplicates\t Size:(%d, %d)'%(data.shape[0], data.shape[1]))
data = data.T.drop_duplicates().T
print('After removing duplicates\t Size:(%d, %d)'%(data.shape[0], data.shape[1]))
print('-----------------------------------------------------------------')

-----------------------------------------------------------------
Before removing duplicates	 Size:(1001, 201)
After removing duplicates	 Size:(1001, 189)
-----------------------------------------------------------------


## Format Ajustment

This code standardizes variable names by converting them to lowercase and using dots as separation characters.

In [4]:
data.columns= data.columns.str.lower()
original_feat = list(data.columns)
for i in range(len(original_feat)):
    data.rename(columns={original_feat[i]: original_feat[i].replace('_','.')}, inplace=True)
    data.rename(columns={original_feat[i]: original_feat[i].replace(' ','.')}, inplace=True) 

original_feat = list(data.columns)
for i in range(len(original_feat)):
    data.rename(columns={original_feat[i]: original_feat[i].replace('...','.')}, inplace=True)
    data.rename(columns={original_feat[i]: original_feat[i].replace('..','.')}, inplace=True)

## Data Preprocessing

n general, measurements and statistics are not always stored correctly, leading to missing values (null cells) in the dataset. A column with many null values does not provide meaningful information for data analysis or machine learning models, leading to inaccurate estimations and nonfunctional models. Thus, we remove columns with more than 1% missing values because such columns would provide insufficient information for analysis. For the remaining columns with missing values, we fill in the gaps using the median of the column. The median is chosen because it is less affected by extreme values (outliers) and provides a more stable estimate for missing data.

Similarly, columns with the same value in every row do not provide helpful information for analysis, as they lack variability, which is critical for identifying patterns in machine learning models. Therefore, we remove any columns with a unique value across all rows.

In [5]:
data1 = data.copy()
target_kpi = 'transfer.datarate'
kpi_aux = data1[target_kpi] 

data1 = data1.loc[:,data1.columns[data1.columns != 'unnamed:.0']]
print('Input dataset \t\t\t\t Size: (%d, %d)'%(data1.shape[0], data1.shape[1]))

# Remove cols with more than 1% of NA
data1 = data1.loc[:,data1.columns[data1.columns != target_kpi]]
data1 = data1.replace([np.inf, -np.inf], np.nan)
data1 = data1.loc[:, data1.columns[data1.isna().mean(axis=0) < 0.01]]
# Replace remaining null cells with the column median
data1 = data1.fillna(data1.median())
print('Remove cols with more than 1%% of Nans \t Size: (%d, %d)'%(data1.shape[0], data1.shape[1]))

# Remove constant columns
data2 = data1.loc[:, (data1.std(axis=0) != 0).values]
print('Remove cols with a single value \t Size: (%d, %d)'%(data2.shape[0], data2.shape[1]))

Input dataset 				 Size: (1001, 189)
Remove cols with more than 1% of Nans 	 Size: (1001, 180)
Remove cols with a single value 	 Size: (1001, 179)


## Multicollinearity Detection and Removal

At this step, the procedure eliminates columns with **multicollinearity**, a limitation that arises when two or more independent variables in the dataset are highly correlated. This limitation can distort the results of a regression model, leading to unreliable coefficient estimates and, consequently, distorting the individual effects of the input features on the model.

To address multicollinearity, we have developed an algorithm that removes columns one by one based on their **variance inflation factor (VIF).** At each iteration, the algorithm identifies the column with the highest VIF and removes it. This process continues until all remaining columns have a VIF below a predefined threshold, which we have set at 1000 in our case. By calculating VIF, we can effectively identify and remove columns with high multicollinearity, thereby ensuring that our model is more accurate and reliable. It's worth noting that this algorithm may take several minutes to complete.

Finally, We save the clean dataset in a CSV file.

In [6]:
vif_vector = np.array([variance_inflation_factor(data2.values, i) for i in range(len(data2.columns))]).reshape(-1,1)
k = 1
while (np.max(vif_vector) > 1000):
    col2rem = data2.columns[np.argmax(vif_vector)]
    print('%d & %s & %E \\\\'%(k, col2rem, np.max(vif_vector)))
    data2 = data2.loc[:, data2.columns != col2rem]
    vif_vector = np.array([variance_inflation_factor(data2.values, i) for i in range(len(data2.columns))]).reshape(-1,1)
    k += 1

data3 = data2.loc[:, data2.columns[~np.isinf(vif_vector).reshape(-1)]]
data3[target_kpi] = kpi_aux
data3.to_csv('data/Op1_CapDL_NoSteps.csv')

data3.shape

1 & unique.bytes.sent.b2a & INF \\
2 & abs.theoricalmaxtp.min & 6.085945E+13 \\
3 & pure.acks.sent.a2b & 1.524061E+13 \\
4 & total.packets.b2a & 2.721208E+13 \\
5 & abs.uplinkdeltadelay.50. & 7.198849E+11 \\
6 & abs.uplinkdelay.min & 2.273971E+12 \\
7 & abs.downlinkdeltadelay.max & 8.445569E+10 \\
8 & abs.uplinkdeltadelay.max & 5.925789E+12 \\
9 & abs.uplinkdeltadelay.25. & 1.153896E+11 \\
10 & abs.downlinkdeltadelay.75. & 6.609624E+10 \\
11 & abs.uplinkdeltadelay.75. & 3.737319E+10 \\
12 & abs.downlinkdelay.avg & 3.882012E+10 \\
13 & abs.downlinkdeltadelay.50. & 1.664273E+09 \\
14 & sack.pkts.sent.a2b & 1.462716E+09 \\
15 & initial.window.bytes.a2b & 1.271719E+09 \\
16 & abs.downlinkdelay.25. & 1.787288E+09 \\
17 & actual.data.bytes.a2b & 2.882397E+08 \\
18 & total.packets.a2b & 1.911827E+08 \\
19 & max.segm.size.a2b & 1.375031E+08 \\
20 & truncated.packets.b2a & 7.531147E+07 \\
21 & rtt.min.last.a2b & 2.620620E+07 \\
22 & truncated.data.b2a & 2.440666E+07 \\
23 & avg.win.adv.b2a & 2.

(1001, 134)